# DermaGen · 02 — Entraînements sur le GPU de Kaggle

Ce notebook ne contient pas de code de modèle : il récupère le dépôt GitHub du projet, lance les scripts de `src/` sur le GPU, puis emballe les résultats pour les télécharger. Tu le réutilises chaque semaine, en ajoutant les nouvelles commandes.

### Réglages à faire une fois dans Kaggle (panneau de droite, « Settings »)

1. **Accelerator** : `GPU T4 x2` ou `GPU P100`. Un seul GPU suffit.
2. **Internet** : activé. Il faut avoir vérifié son numéro de téléphone sur Kaggle.
3. **Add Input** : chercher **Skin Cancer MNIST: HAM10000** (auteur kmader) et l'ajouter.

### À savoir

- Le dossier `/kaggle/working` est vidé à la fin de chaque session : **télécharge l'archive de résultats** (dernière cellule) avant de fermer.
- Une session GPU dure au maximum 12 heures ; le quota est d'environ 30 heures de GPU par semaine.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Récupérer le code du projet

Remplace `<ton-pseudo>` par ton nom d'utilisateur GitHub. Si le dépôt existe déjà dans la session, la cellule récupère simplement la dernière version.

*Dépôt privé :* dans Kaggle, menu **Add-ons → Secrets**, crée un secret `GITHUB_TOKEN` (un token GitHub en lecture seule), puis mets `PRIVATE = True`.

In [ ]:
import os
from pathlib import Path

GITHUB_USER = "<ton-pseudo>"
PRIVATE = False

url = f"https://github.com/{GITHUB_USER}/dermagen.git"
if PRIVATE:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{GITHUB_USER}/dermagen.git"

REPO = Path("/kaggle/working/dermagen")
if REPO.exists():
    !git -C {REPO} pull -q
else:
    !git clone -q {url} {REPO}
os.chdir(REPO)
!git log -1 --format="Dernier commit : %h %s (%cr)"

## 3. Installer les dépendances et vérifier les données

PyTorch est déjà installé sur Kaggle ; pip n'ajoute que ce qui manque. `src.config` trouve tout seul le dataset ajouté en entrée : toutes les lignes doivent indiquer **OK**.

In [ ]:
!pip install -q -r requirements.txt
!python -m src.config

## 4. Préparer les données

À refaire à chaque nouvelle session, car `/kaggle/working` est vidé (2 à 3 minutes). Ajoute `--sizes 28 64` le jour où tu passes au 64×64.

In [ ]:
!python -m src.prepare_data

## 5. Semaine 1 : le classifieur

Les baselines de la question Q3, avec trois graines chacune (0, 1, 2) pour rapporter moyenne ± écart-type. Chaque run ajoute une ligne à `results/classifier_runs.csv`.

Compte quelques minutes par run sur GPU.

In [ ]:
for seed in (0, 1, 2):
    !python -m src.train_classifier --seed {seed}
    !python -m src.train_classifier --weighted --seed {seed}
    !python -m src.train_classifier --balanced --augment --seed {seed}

**L'arbitre.** Choisis la variante qui a la meilleure balanced accuracy *sur la validation* (pas sur le test), puis relance-la avec `--export` : elle est copiée dans `weights/classifier.pt`, que les métriques et la démo utiliseront.

In [ ]:
!python -m src.train_classifier --weighted --augment --seed 0 --export

## 6. Résumé des runs

In [ ]:
import pandas as pd

runs = pd.read_csv("results/classifier_runs.csv")
runs["variant"] = runs["run"].str.replace(r"_s\d+", "", regex=True)
summary = (runs.groupby("variant")[["val_balanced_accuracy", "test_balanced_accuracy", "test_macro_f1",
                                    "test_recall_df", "test_recall_vasc", "minutes"]]
               .agg(["mean", "std"]).round(3))
summary

## 7. Semaine 2 : le cVAE

**Avant de lancer :** l'arbitre `weights/classifier.pt` doit exister (section 5), car les métriques l'utilisent. Si tu reprends une nouvelle session, relance d'abord les sections 2 à 4, puis copie ton `weights/classifier.pt` depuis GitHub (il est versionné) ou réexporte-le.

### 7.1 Ablations, graine 0

Écris tes hypothèses dans `JOURNAL.md` **avant** de lancer. Compter environ 3 à 5 minutes par run sur GPU.

In [ ]:
# beta ablation, latent size fixed at 16
for beta in (0.5, 1, 2, 4):
    !python -m src.train_cvae --beta {beta} --latent 16 --seed 0

# latent size ablation, beta fixed at 1 (z = 2 also gives the 2-D map of the demo)
for z in (2, 8, 32):
    !python -m src.train_cvae --beta 1 --latent {z} --seed 0

### 7.2 Générer avec le meilleur checkpoint de chaque run

Même mélange de classes que le test (option par défaut), même graine pour tous : la comparaison est équitable.

In [ ]:
from pathlib import Path

for ckpt in sorted(Path("checkpoints").glob("cvae_*_best.pt")):
    !python -m src.sample --model cvae --checkpoint {ckpt} --seed 0

### 7.3 Évaluer

`--cas` entraîne un classifieur par fichier sur les seules images générées : c'est plus long, mais c'est la mesure la plus parlante pour Q3. La ligne `real_train` sert de plancher : ce qu'obtiendraient de vraies images.

In [ ]:
!python -m src.evaluate --pattern "cvae_*" --cas

In [ ]:
import pandas as pd

m = pd.read_csv("results/metrics.csv")
cols = ["run", "fid_domain", "fid_inception", "precision", "recall", "fidelity", "nn_ratio",
        "cas_balanced_accuracy", "ms_per_image"]
m[[c for c in cols if c in m.columns]].sort_values("fid_domain")

### 7.4 Confirmer le meilleur réglage avec 3 graines, puis l'exporter

Remplace `BETA` et `Z` par le réglage retenu (en t'appuyant surtout sur `fid_domain`, `recall` et `cas`). On ne réentraîne pas la graine 0 : son checkpoint existe déjà.

In [ ]:
BETA, Z = 1, 16
tag = f"b{BETA:g}_z{Z}"          # same naming as train_cvae.py, e.g. b1_z16

for seed in (1, 2):
    ckpt = f"checkpoints/cvae_s{seed}_{tag}_best.pt"
    !python -m src.train_cvae --beta {BETA} --latent {Z} --seed {seed}
    !python -m src.sample --model cvae --checkpoint {ckpt} --seed 0
pattern = f"cvae_s*_{tag}_seed*"
!python -m src.evaluate --pattern "{pattern}" --cas

# Export the seed-0 model for the demo (no retraining needed)
from src import config, utils
from src.sample import load_generator
best = load_generator("cvae", f"checkpoints/cvae_s0_{tag}_best.pt")
print(utils.export_weights(best, config.WEIGHTS_DIR / "cvae.pt", best.config))

## 8. Semaines suivantes

Les commandes s'ajouteront ici au fur et à mesure que les scripts seront écrits.

In [ ]:
# Week 3: GAN
# !python -m src.train_cgan --loss bce --seed 0

# Week 4: diffusion
# !python -m src.train_ddpm --schedule linear --seed 0

# Week 5: augmentation experiment
# !python -m src.augment

## 9. Emballer les résultats pour les télécharger

L'archive contient `weights/`, `results/` et les checkpoints. Pour la récupérer : panneau de droite → **Output** → bouton de téléchargement à côté du fichier `.zip`. Dézippe-la ensuite à la racine du projet sur ton ordinateur, puis fais un commit (sans les checkpoints, que `.gitignore` exclut).

In [ ]:
from datetime import datetime

archive = f"/kaggle/working/dermagen_outputs_{datetime.now():%Y%m%d_%H%M}.zip"
!cd {REPO} && zip -qr {archive} weights results checkpoints -x "*.gitkeep"
!ls -lh {archive}